In [ ]:
import numpy as np
import cupy as cp
import matplotlib.pyplot as plt
from NN_helper_functions import run_model, predict
from process_data_higgs import preprocess_data

In [ ]:
X_train, Y_train, X_test, Y_test = preprocess_data()

X_train_gpu = cp.asarray(X_train)
Y_train_gpu = cp.asarray(Y_train)
X_test_gpu = cp.asarray(X_test)
Y_test_gpu = cp.asarray(Y_test)

In [ ]:
parameters = {}
costs = {}
layers_dims = [X_train.shape[0], 20, 10, 7, 4, 1]
learning_rates = [0.01, 0.025, 0.05, 0.075, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]

In [ ]:
for rate in learning_rates:
    parameters[rate], costs[rate] = run_model(X_train_gpu, Y_train_gpu, layers_dims, num_iterations=10000, learning_rate=rate, print_cost=True)

In [ ]:
accuracies = {}
for rate in learning_rates:
    print(f"Learning rate: {rate}")
    print("Train: ")
    train_mat, train_acc = predict(X_train_gpu, Y_train_gpu, parameters[rate])
    print("Test: ")
    test_mat, test_acc = predict(X_test_gpu, Y_test_gpu, parameters[rate])
    print()
    accuracies[rate] = {"train": train_acc, "test": test_acc}

In [ ]:
plt.figure(figsize=(8, 5))

for lr, cost_list in costs.items():
    cost_values = [float(c) for c in cost_list]
    plt.plot(cost_values, label=f"lr={lr}")

plt.xlabel("Iterations (x100)")
plt.ylabel("Cost")
plt.title("Cost convergence across learning rates")
plt.legend()
plt.show()

In [ ]:
majority_class = Y_train.mean() > 0.5
baseline_accuracy = max(Y_train.mean(), 1 - Y_train.mean())
print("Majority baseline accuracy:", baseline_accuracy)

In [ ]:
specs = list(accuracies.keys()) 
train_acc = [accuracies[spec]["train"] for spec in specs]
test_acc = [accuracies[spec]["test"] for spec in specs]

train_acc = [float(a) for a in train_acc]
test_acc = [float(a) for a in test_acc]

x = np.arange(len(specs))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 8))
bars1 = ax.bar(x - width/2, train_acc, width, label='Train accuracy')
bars2 = ax.bar(x + width/2, test_acc, width, label='Test accuracy')

ax.set_xlabel('Learning rate')
ax.set_ylabel('Accuracy')
ax.set_title('Train vs test accuracy by learning rate')
ax.set_xticks(x)
ax.set_xticklabels(specs, rotation=20, ha='right')
ax.set_ylim(0.6, 0.87)  # wide range to accommodate the "oversized" collapse
ax.legend()

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords="offset points", ha='center', fontsize=7, rotation=90)

plt.tight_layout()
plt.show()